Licensed under the Apache License, Version 2.0

In [ ]:
!pip install pycontrails

In [ ]:
import collections
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import matplotlib
matplotlib.rcParams['figure.figsize'] = [9, 6]
matplotlib.rcParams['font.size'] = 18

from pycontrails.physics import thermo


In [ ]:
!wget https://zenodo.org/records/19440374/files/paper_release_dataset.zip?download=1 -O paper_release_dataset.zip
!unzip paper_release_dataset.zip
!rm paper_release_dataset.zip

In [ ]:
with open('paper_release_dataset.pq', 'rb') as f:
  full_df = pd.read_parquet(f)

In [ ]:
full_df

In [ ]:
def get_fpr(tpr, precision, flight_fraction):
  return flight_fraction * tpr * (1-precision) / (precision * (1 - flight_fraction))

TPR=0.33
PRECISON=0.70
FRACTION=0.05
FPR = get_fpr(TPR, PRECISON, FRACTION)
print('fpr:', FPR)

def get_hybrid_metric(is_match, metric, tpr=TPR, fpr=FPR, min_metric=0):
  metric = np.clip(metric, min_metric, None)
  return np.where(
      is_match,
      tpr*metric / (tpr * metric + fpr * (1-metric)),
      (1-tpr)*metric/((1-tpr)*metric + (1-fpr) * (1-metric)))



In [ ]:

def get_pr(p1, p2):
  a = np.sum(p1 & p2)
  b = np.sum(p1 & ~p2)
  c = np.sum(~p1 & p2)
  precision = a / (a + b)
  recall = a / (a + c)
  return precision, recall

def get_pr_curve(metric, actual, cutoffs = np.linspace(0, 1, num=11, endpoint=True)):
  ps = []
  rs = []
  for i in cutoffs:
    p, r = get_pr(metric > i, actual)
    ps.append(p)
    rs.append(r)
  return np.array(ps), np.array(rs)

def get_ets(p1, p2):
  a = np.sum(p1 & p2)
  b = np.sum(p1 & ~p2)
  c = np.sum(~p1 & p2)
  d = np.sum(~p1 & ~p2)
  r = (a + b) * (a + c) / (a + b + c +d)
  return (a - r) / (a + b + c - r)

def find_close(tdf, dist_threshold, alt_threshold, time_threshold):
  mask = (
        (tdf.s2_distance < dist_threshold) &
        (tdf.altitude_diff < alt_threshold) &
        (tdf.time_diff < time_threshold)
  )
  match_df = tdf[['insitu_id', 'match']][mask].groupby('insitu_id').mean()
  output_df = tdf.drop(columns='match').groupby('insitu_id').first()
  return output_df.merge(match_df, on='insitu_id', how='left').reset_index()

def eval_close(tdf, dist_threshold, alt_threshold, time_threshold, rhi_key):
  one_row_df = find_close(tdf, dist_threshold, alt_threshold, time_threshold)
  mask = ~one_row_df.match.isna()
  p, r= get_pr(one_row_df[mask].match > 0.5, one_row_df[mask][rhi_key] > 1)
  return p, r, np.sum(mask), np.sum(one_row_df[mask][rhi_key] > 1)

def find_close_and_filter(tdf, dist_threshold, alt_threshold, time_threshold):
  close_df = find_close(tdf, dist_threshold, alt_threshold, time_threshold)
  mask = ~close_df.match.isna()
  close_df = close_df[mask]
  return close_df

def train_validation_split(tdf):
  dayofyear = pd.to_datetime(tdf.timestamp, unit='s').dt.dayofyear
  train_df = tdf[dayofyear % 6 == 0]
  validation_df = tdf[dayofyear % 6 != 0]
  return train_df, validation_df


In [ ]:
train_df, validation_df = train_validation_split(full_df)

# try to find good closeness parameters

In [ ]:
rows = []
for dist_threshold in [5, 10, 15, 30, 50, 60]:
  for alt_threshold in [10, 20, 30, 50, 100, 300]:
    for time_threshold in [120, 300, 600, 1800, 3600]:
      p, r, n1, n2 = eval_close(train_df, dist_threshold, alt_threshold, time_threshold, 'iagos_rhi')
      rows.append({'dist_threshold': dist_threshold, 'alt_threshold': alt_threshold, 'time_threshold': time_threshold, 'precision': p, 'recall': r, 'n': n1, 'npos': n2})
full_results_df = pd.DataFrame(rows)
full_results_df = full_results_df.sort_values('recall', ascending=False)


In [ ]:
results_df = full_results_df[full_results_df.n > 30_000]
results_df.head(20)

In [ ]:
plt.scatter(results_df.recall, results_df.precision)
plt.scatter(results_df.recall.values[0:1], results_df.precision.values[0:1], marker='*', s=100)
plt.xlabel('TPR')
plt.ylabel('precision')
plt.xlim([0.2, 0.32])
plt.ylim([0.52, 0.62])
plt.grid(True)


In [ ]:
BEST_CLOSENESS_PARAMS = {'dist_threshold': 15, 'alt_threshold': 20, 'time_threshold': 300}

# plot of scores

In [ ]:
metric = np.linspace(0, 1, num=51, endpoint=True)
plt.plot(metric, metric, label='$S_{ensemble~mean}$')
plt.plot(metric, get_hybrid_metric(True, metric), label='$S_{hybrid(observed)}$')
plt.plot(metric, get_hybrid_metric(False, metric), label='$S_{hybrid(not~observed)}$')
plt.xlabel('$S_{ensemble~mean}$')
plt.legend()
plt.grid(True)


# filter dataset

In [ ]:
close_eval_df = find_close_and_filter(validation_df, **BEST_CLOSENESS_PARAMS)

In [ ]:
t_fields = [f'era5_ens_modellevel_t_{i}' for i in range(10)] + ['arco_era5_model_level_t']
ts_we_use = close_eval_df[t_fields].values
bad_ts = np.any(np.isnan(ts_we_use) | (ts_we_use < 100) | (ts_we_use > 400), axis=-1)
print(np.mean(bad_ts))
close_eval_df = close_eval_df[~bad_ts]

In [ ]:
tdf = close_eval_df
close_eval_df

In [ ]:
len(np.unique(close_eval_df.insitu_id)), len(close_eval_df)

# compute schmidt appleman

In [ ]:
RHI_ADJ = 0.97
def compute_pycontrails_sac(
    ambient_temperature, specific_humidity, ambient_air_pressure
):
  if len(ambient_air_pressure.shape) == 1 and len(ambient_temperature.shape) == 2:
    ambient_air_pressure = ambient_air_pressure[:, np.newaxis]

  relative_humidity = thermo.rhi(specific_humidity, ambient_temperature, ambient_air_pressure)
  sac_pc = humidity_utils.schmidt_appleman_ecmwf(ambient_temperature, ambient_air_pressure, specific_humidity=specific_humidity)
  return sac_pc, relative_humidity

s, rhi = compute_pycontrails_sac(tdf.iagos_t, tdf.iagos_q, tdf.pressure*100)
tdf['iagos_rhi'] = rhi
tdf['iagos_sac'] = s
tdf['iagos_pos'] = s & (rhi > 1)

s, rhi = compute_pycontrails_sac(tdf.arco_era5_model_level_t, tdf.arco_era5_model_level_q/RHI_ADJ, tdf.pressure*100)
tdf['arco_rhi'] = rhi
tdf['arco_sac'] = s
tdf['arco_pos'] = s & (rhi > 1)

s, rhi = compute_pycontrails_sac(tdf[[f'era5_ens_modellevel_t_{i}' for i in range(10)]].values, tdf[[f'era5_ens_modellevel_q_{i}' for i in range(10)]].values/RHI_ADJ, tdf.pressure.values*100)
era_ensemble_rhi = rhi
era_ensemble_sac = s
era_ensemble_pos = s & (rhi > 1)

s, rhi = compute_pycontrails_sac(tdf.ecmwf_historical_t, tdf.ecmwf_historical_q/RHI_ADJ, tdf.pressure*100)
tdf['hres_rhi'] = rhi
tdf['hres_sac'] = s
tdf['hres_pos'] = s & (rhi > 1)

s, rhi = compute_pycontrails_sac(tdf[[f'ecmwf_hres_ensemble_t_{i}' for i in range(50)]].values, tdf[[f'ecmwf_hres_ensemble_q_{i}' for i in range(50)]].values/RHI_ADJ, tdf.pressure.values[:, np.newaxis]*100)
hres_ensemble_rhi = rhi
hres_ensemble_sac = s
hres_ensemble_pos = s & (rhi > 1)

s, rhi = compute_pycontrails_sac(tdf.arco_era5_model_level_t_coarse.values,tdf.arco_era5_model_level_q_coarse.values/RHI_ADJ, tdf.pressure.values*100)
tdf['arco_rhi_coarse'] = rhi
tdf['arco_sac_coarse'] = s
tdf['arco_pos_coarse'] = s & (rhi > 1)

# Main S_hybrid figure

In [ ]:
#@title figure for paper
plt.figure(figsize=(16, 8))
rhi_threshold=1

hres_mask = ~np.isnan(tdf.ecmwf_hres_ensemble_t_0)

# era5
ax1 = plt.subplot(121)
nominal_p, nominal_r = get_pr(tdf.arco_pos, tdf.iagos_pos)
f = ax1.plot(nominal_r, nominal_p, 's', label='$S_{nominal}$')

nominal_p, nominal_r = get_pr(tdf.arco_pos_coarse, tdf.iagos_pos)
f = ax1.plot(nominal_r, nominal_p, 'o', label='$S_{nominal}(coarse)$', color=f[0].get_color())

metric = np.mean(era_ensemble_pos, axis=1)
era_p, era_r = get_pr_curve(metric, tdf.iagos_pos, cutoffs=np.linspace(0, 0.85, num=10+1, endpoint=True))
ax1.plot(era_r, era_p, '.-', label='$S_{ensemble~mean}$', color=f[0].get_color())
new_metric = get_hybrid_metric(tdf.match>0, metric)
hybrid3_p, hybrid3_r = get_pr_curve(new_metric, tdf.iagos_pos, cutoffs=np.linspace(0, 1, num=10+1, endpoint=True))
ax1.plot(hybrid3_r, hybrid3_p, '--', label='$S_{hybrid}$', color=f[0].get_color())

ax1.legend()
ax1.grid()
ax1.set_xlabel('TPR')
ax1.set_ylabel('precision')
ax1.set_title('ERA5')
ax1.text(0.05, 0.1, '(a)', transform=ax1.transAxes)
# ax1.set_xlim([0.2, 0.75])
# ax1.set_ylim([0.35, 0.65])

ax2 = plt.subplot(122)
nominal_p, nominal_r = get_pr(tdf[hres_mask].hres_pos, tdf[hres_mask].iagos_pos)
f = ax2.plot(nominal_r, nominal_p, 's', label='$S_{nominal}$')

metric = np.mean(hres_ensemble_pos[hres_mask][:, :10], axis=1)
era_p, era_r = get_pr_curve(metric, tdf[hres_mask].iagos_pos, cutoffs=np.linspace(0, 0.8, num=10+1, endpoint=True))
ax2.plot(era_r, era_p, '.-', label='$S_{ensemble~mean}(10)$', color=f[0].get_color())
new_metric = get_hybrid_metric(tdf[hres_mask].match>0, metric)
hybrid3_p, hybrid3_r = get_pr_curve(new_metric, tdf[hres_mask].iagos_pos, cutoffs=np.linspace(0, 1, num=10+1, endpoint=True))
ax2.plot(hybrid3_r, hybrid3_p, '--', label='$S_{hybrid}(10)$', color=f[0].get_color())

metric = np.mean(hres_ensemble_pos[hres_mask], axis=1)
era_p, era_r = get_pr_curve(metric, tdf[hres_mask].iagos_pos, cutoffs=np.linspace(0, 0.8, num=50+1, endpoint=True))
f = ax2.plot(era_r, era_p, '.-', label='$S_{ensemble~mean}(50)$',)
new_metric = get_hybrid_metric(tdf[hres_mask].match>0, metric)
hybrid3_p, hybrid3_r = get_pr_curve(new_metric, tdf[hres_mask].iagos_pos, cutoffs=np.linspace(0, 1, num=50+1, endpoint=True))
ax2.plot(hybrid3_r, hybrid3_p, '--', label='$S_{hybrid}(50)$', color=f[0].get_color())

ax2.legend(loc='upper right')
ax2.grid()
ax2.set_title('IFS forecast')
ax2.set_xlabel('TPR')
ax2.set_ylabel('precision')
ax2.text(0.05, 0.1, '(b)', transform=ax2.transAxes)







# AUC diffs

In [ ]:
import scipy
RES = 0.01
def interpolate(p, r):
  nan_mask = np.isnan(p) | np.isnan(r)
  p = p[~nan_mask]
  r = r[~nan_mask]
  new_rs = np.arange(np.min(r), np.max(r), RES)
  new_ps = scipy.interpolate.interp1d(r, p, bounds_error=False, fill_value='extrapolate')(new_rs)
  return new_ps, new_rs

def _auc(p):
  return np.sum(p) * RES

def comp_auc_diff(p1, r1, p2, r2):
  new_p1, new_r1 = interpolate(p1, r1)
  new_p2, new_r2 = interpolate(p2, r2)

  max_r = min(np.max(new_r1), np.max(new_r2))
  min_r = max(np.min(new_r1), np.min(new_r2))
  mask1 = (new_r1>=min_r) & (new_r1<=max_r)
  new_p1 = new_p1[mask1]
  new_r1 = new_r1[mask1]
  mask2 = (new_r2>=min_r) & (new_r2<=max_r)
  new_p2 = new_p2[mask2]
  new_r2 = new_r2[mask2]
  return (_auc(new_p2) - _auc(new_p1))/(max_r - min_r)

def bootstrap_auc_diff(df, metric, truth, n_bootstraps=100, frac_threshold=0):
  dt = pd.to_datetime(df.timestamp, unit='s')
  year_day = dt.dt.dayofyear + dt.dt.year*1000
  unique_days = np.unique(year_day)
  aucs = []
  for _ in range(n_bootstraps):
    days = np.random.choice(unique_days, len(unique_days), replace=True)
    mask = np.isin(year_day, days)
    ttdf = df[mask]
    era_p, era_r = get_pr_curve(metric[mask], truth[mask], cutoffs=np.linspace(0, 0.8, num=10+1, endpoint=True))
    new_metric = get_hybrid_metric(ttdf.match>frac_threshold, metric[mask], ttdf.altitude)
    hybrid3_p, hybrid3_r = get_pr_curve(new_metric, truth[mask], cutoffs=np.linspace(0, 1, num=10+1, endpoint=True))
    aucs.append(comp_auc_diff(era_p, era_r, hybrid3_p, hybrid3_r))
  print(f'{np.mean(aucs):.3f} +/- {np.std(aucs):.3f}')

In [ ]:
bootstrap_auc_diff(tdf, np.mean(era_ensemble_pos, axis=1), tdf.iagos_pos)

In [ ]:
bootstrap_auc_diff(tdf[hres_mask], np.mean(hres_ensemble_pos[hres_mask, :10], axis=1), tdf[hres_mask].iagos_pos)

In [ ]:
bootstrap_auc_diff(tdf[hres_mask], np.mean(hres_ensemble_pos[hres_mask], axis=1), tdf[hres_mask].iagos_pos)

In [ ]:
bootstrap_auc_diff(tdf, np.mean(era_ensemble_rhi > 1, axis=1), tdf.iagos_rhi > 1)

# TPR/FPR sensitivity analysis

In [ ]:
plt.figure(figsize=(16, 7))

COATSAC_LINEARIZED_TPR = 0.62
COATSAC_LINEARIZED_PRECISION = 0.72
CONTRAIL_LINEARIZED_FLIGHT_FRACTION = 0.027/0.48 # 0.48 comes from differences in cocip numbers before and after filtering
COATSAC_LINEARIZED_FPR = get_fpr(COATSAC_LINEARIZED_TPR, COATSAC_LINEARIZED_PRECISION, CONTRAIL_LINEARIZED_FLIGHT_FRACTION)
print('coatsac fpr', COATSAC_LINEARIZED_FPR)
ALL_TPR = 0.18
ALL_PRECISION = 0.91
ALL_FLIGHT_FRACTION = 0.115
ALL_FPR = get_fpr(ALL_TPR, ALL_PRECISION, CONTRAIL_LINEARIZED_FLIGHT_FRACTION)
print('all fpr', ALL_FPR)

ax1 = plt.subplot(121)
metric = np.linspace(0, 1, num=11, endpoint=True)
ax1.plot(metric, metric, label='$S_{ensemble~mean}$')
f=ax1.plot(metric, get_hybrid_metric(True, metric), label='all contrails')
ax1.plot(metric, get_hybrid_metric(False, metric), '--', color=f[0].get_color())
f=ax1.plot(metric, get_hybrid_metric(True, metric, tpr=COATSAC_LINEARIZED_TPR, fpr=COATSAC_LINEARIZED_FPR), label='linearized contrails')
ax1.plot(metric, get_hybrid_metric(False, metric, tpr=COATSAC_LINEARIZED_TPR, fpr=COATSAC_LINEARIZED_FPR), '--', color=f[0].get_color())
f=ax1.plot(metric, get_hybrid_metric(True, metric, tpr=ALL_TPR, fpr=ALL_FPR), label='bad observations')
ax1.plot(metric, get_hybrid_metric(False, metric, tpr=ALL_TPR, fpr=ALL_FPR), '--', color=f[0].get_color())
ax1.set_xlabel('$S_{ensemble~mean}$')
ax1.legend()
ax1.grid(True)
ax1.text(0.05, 0.9, '(a)', transform=ax1.transAxes)

ax2 = plt.subplot(122)
# era5
nominal_p, nominal_r = get_pr(tdf.arco_pos, tdf.iagos_pos)
f = plt.plot(nominal_r, nominal_p, 's', label='era nominal')

metric = np.mean(era_ensemble_pos, axis=1)
era_p, era_r = get_pr_curve(metric, tdf.iagos_pos, cutoffs=np.linspace(0, 0.8, num=10+1, endpoint=True))
ax2.plot(era_r, era_p, '.-', label='era ensemble', color=f[0].get_color())

def plot_with_metrics(tpr, fpr, label):
  new_metric = get_hybrid_metric(tdf.match>0, metric, tpr=tpr, fpr=fpr)
  hybrid3_p, hybrid3_r = get_pr_curve(new_metric, tdf.iagos_pos, cutoffs=np.linspace(0, 1, num=10+1, endpoint=True))
  ax2.plot(hybrid3_r, hybrid3_p, '--', label=label)

plot_with_metrics(TPR, FPR, 'warming contrails')
plot_with_metrics(COATSAC_LINEARIZED_TPR, COATSAC_LINEARIZED_FPR, 'linearized contrails')
plot_with_metrics(ALL_TPR, ALL_FPR, 'all contrails')

ax2.legend()
ax2.grid(True)
ax2.set_xlabel('TPR')
ax2.set_ylabel('precision')
ax2.text(0.05, 0.1, '(b)', transform=ax2.transAxes)



# threshold sensitivity analysis

In [ ]:
results_df.head(30)

In [ ]:
interesting_points = np.array([67, 126, 76]) # time, distance


In [ ]:
plt.scatter(results_df.recall, results_df.precision)
plt.scatter(results_df.recall.values[:1], results_df.precision.values[:1], marker='*', s=100)
plt.scatter(results_df.recall.loc[interesting_points], results_df.precision.loc[interesting_points], marker='+', s=100)
plt.xlabel('tpr')
plt.ylabel('precision')
# plt.xlim([0.2, 0.35])
# plt.ylim([0.45, 0.55])
plt.grid(True)


In [ ]:
for pt in interesting_points:
  row = results_df.loc[pt]
  print(row.dist_threshold, row.alt_threshold, row.time_threshold)

In [ ]:
thresh_dfs = {0: tdf}
thresh_ensemble_pos = {0:era_ensemble_pos}
for pt in interesting_points:
  row = results_df.loc[pt]
  thresh_df = find_close_and_filter(validation_df, dist_threshold=row.dist_threshold, alt_threshold=row.alt_threshold, time_threshold=row.time_threshold)
  t_fields = [f'era5_ens_modellevel_t_{i}' for i in range(10)] + ['arco_era5_model_level_t']
  ts_we_use = thresh_df[t_fields].values
  bad_ts = np.any(np.isnan(ts_we_use) | (ts_we_use < 100) | (ts_we_use > 400), axis=-1)
  print(np.mean(bad_ts))
  thresh_df = thresh_df[~bad_ts]

  s, rhi = compute_pycontrails_sac(thresh_df.iagos_t, thresh_df.iagos_q, thresh_df.pressure*100)
  thresh_df['iagos_pos'] = s & (rhi > 1)

  s, rhi = compute_pycontrails_sac(thresh_df.arco_era5_model_level_t,thresh_df.arco_era5_model_level_q/RHI_ADJ, thresh_df.pressure*100)
  thresh_df['arco_pos'] = s & (rhi > 1)

  s, rhi = compute_pycontrails_sac(thresh_df[[f'era5_ens_modellevel_t_{i}' for i in range(10)]].values, thresh_df[[f'era5_ens_modellevel_q_{i}' for i in range(10)]].values/RHI_ADJ, thresh_df.pressure.values[:, np.newaxis]*100)
  pos = s & (rhi > 1)
  thresh_dfs[pt] = thresh_df
  thresh_ensemble_pos[pt] =pos




In [ ]:

plt.figure(figsize=(12, 6))
labels = ['Base', 'Time', 'Distance', 'Altitude']
ax0 = plt.subplot(121)
ax0.scatter(results_df.recall, results_df.precision)
f = {}
f[0] = ax0.scatter(results_df.recall.values[:1], results_df.precision.values[:1], marker='*', s=100, label='Base')
for pt, case in zip(interesting_points, labels[1:]):
  f[pt] = ax0.scatter(results_df.recall.loc[pt], results_df.precision.loc[pt], marker='+', s=100)
ax0.set_xlabel('TPR')
ax0.set_ylabel('precision')
plt.xlim([0.2, 0.32])
plt.ylim([0.52, 0.62])
ax0.grid(True)
ax0.text(0.05, 0.1, '(a)', transform=ax0.transAxes)


# era5
ax1 = plt.subplot(122)
for pt, case in zip(np.concatenate([[0], interesting_points]), labels):
  if pt == 0:
    row = results_df.loc[126]
  else:
    row = results_df.loc[pt]
  nominal_p, nominal_r = get_pr(thresh_dfs[pt].arco_pos, thresh_dfs[pt].iagos_pos)
  ax1.plot(nominal_r, nominal_p, 's', label=case,  color=f[pt].get_facecolors()[0])

  metric = np.mean(thresh_ensemble_pos[pt], axis=1)
  era_p, era_r = get_pr_curve(metric, thresh_dfs[pt].iagos_pos, cutoffs=np.linspace(0, 0.8, num=10+1, endpoint=True))
  ax1.plot(era_r, era_p, '.-', color=f[pt].get_facecolors()[0])
  new_metric = get_hybrid_metric(thresh_dfs[pt].match>0, metric)
  hybrid3_p, hybrid3_r = get_pr_curve(new_metric, thresh_dfs[pt].iagos_pos, cutoffs=np.linspace(0, 1, num=10+1, endpoint=True))
  ax1.plot(hybrid3_r, hybrid3_p, '--', color=f[pt].get_facecolors()[0])

ax1.legend()
ax1.grid()
ax1.set_xlabel('TPR')
ax1.set_ylabel('precision')
ax1.text(0.05, 0.9, '(b)', transform=ax1.transAxes)



# Aggregate warming

In [ ]:
with open('paper_release_dataset2.pq', 'rb') as f:
  retro_df = pd.read_parquet(f)

In [ ]:
for column in ['mean_ensemble_cocip_ef', 'nominal_cocip_ef', 'frac_ensembles_making_contrails']:
  retro_df[column] = retro_df[column].fillna(0)

In [ ]:
def bootstrap_estimate(tdf, tpr=TPR, fpr=FPR):
  tdf['hybrid'] = np.where(tdf.frac_ensembles_making_contrails==0, 0, get_hybrid_metric(tdf.best_match_score < 3, tdf.frac_ensembles_making_contrails, tpr=tpr, fpr=fpr) * tdf.mean_ensemble_cocip_ef / tdf.frac_ensembles_making_contrails)

  bootstraps = collections.defaultdict(list)
  for _ in range(100):
    boot_df = tdf.sample(len(tdf), replace=True)
    bootstraps['hybrid'].append(np.sum(boot_df.hybrid * boot_df.segment_length)/np.sum(boot_df.segment_length)/1e6)
    bootstraps['ensemble_mean'].append(np.sum(boot_df.mean_ensemble_cocip_ef * boot_df.segment_length)/np.sum(boot_df.segment_length)/1e6)

  outputs = {}
  for key in bootstraps:
    outputs[f'{key}_val'] = np.mean(bootstraps[key])
    outputs[f'{key}_err'] = np.std(bootstraps[key])
  return outputs



In [ ]:
outputs = bootstrap_estimate(retro_df)
for field in ['hybrid', 'ensemble_mean']:
  print(f'{field}: {outputs[f'{field}_val']:.4f} +/- {outputs[f'{field}_err']:.4f}')

In [ ]:
n_hours = np.arange(24)
retro_df['hour'] = pd.to_datetime(retro_df.original_flight_timestamp, unit='s').dt.hour
ys = collections.defaultdict(list)
for n_hour in n_hours:
  hour_df = retro_df[retro_df.hour ==n_hour]
  outputs = bootstrap_estimate(hour_df)
  for key, val in outputs.items():
    ys[key].append(val)


In [ ]:
plt.plot(n_hours, ys['ensemble_mean_val'], '.-', label='$EF_{ensemble~mean}^{pm}$')
plt.plot(n_hours, ys['hybrid_val'], '.-', label='$EF_{hybrid}^{pm}$')
plt.legend()
plt.grid()
plt.xlabel('hour of day')
plt.ylabel('Contrail forcing (MJ/m)')
plt.tight_layout()
